In [ ]:
import responses as R
import parsers as Parsers
import validator as V
import clean_query_manager as QM
import clean_procedural as CP
import clean_prompt as P
from pprint import pprint
import json
import pyperclip
%load_ext autoreload
%autoreload 2


PROJECT_ID = "Post apocalyptic cyberpunk"
LORE = """A world ravaged by nuclear and biological wars. Survivors leave on the ruins of the old world, but have evolve the cybernetic implants to survive in the new world. Now they live in a world of high technology and low life with rich corporations in nice bunkered cities and poor people in endless savage desert land.  """
GROUP_DESCRIPTION = "Near the city of New York, a gang of bikers roam into the desert and live from pillaging and robbing corporate transports that travel between the cities. They are a small group of 20 people, but they are very well organized and have a lot of weapons. They are led by a charismatic leader who is a former soldier."
WORLD_TYPE = "Post apocalyptic cyberpunk"
INSTRUCTION = f"You are an assistant who generates on-demand data for a tabletop {WORLD_TYPE} role-playing game"
SCALE = "local"
TYPE = "Gang"
POPULATION = 65
SEED = 42
# RACE_RATIO = {"human": 0.6, "half-elf": 0.15, "elf":0.05, "dwarf": 0.15, "halfling": 0.025, "gnome": 0.025}
# potentially all generation parameters (groups distributions, and so on)

# MEAN_GROUP_SIZE = 8 ?????


# Bootstrap data
content = {
    "project_id": PROJECT_ID,
    "generation": {
        "seed": SEED,  # "optional"
        "instruction": INSTRUCTION,  # "optional"
    },
    "setting": {
        "description": LORE,
        "type": WORLD_TYPE
    },
    "details": {
        "type": TYPE,
        "scale": SCALE,  # "optional"
        "population": POPULATION,  # "optional"
        "description": GROUP_DESCRIPTION,
        # "races": RACE_RATIO  # "optional"
    }
}

PROJECT_ID = content["project_id"]

#### BOOTSTRAP ####
bootstrap_prompt = P.bootstrap(content)
pyperclip.copy(INSTRUCTION + "\n" + bootstrap_prompt)
response = R.bootstrap()
parsed_response = Parsers.parse_json(response)
patch = V.bootstrap(parsed_response)
QM.update_dict(content, patch)


#### DESCRIPTION ####
description_prompt = P.description(content)
response = R.description()
parsed_response = Parsers.parse_json(response)
patch = V.descriptions(parsed_response)
QM.update_dict(content, patch)
# print(json.dumps(content, indent=4))


#### DETAILS ####
# test with "culture" only
culture_prompt = P.detail(content, "culture")
response = R.culture()
parsed_response = Parsers.parse_json(response)
patch = V.detail(parsed_response)
QM.update_dict(content, patch)

# TODO Detail Setting

#### EXTERNAL INFLUENCES ####
external_influences_prompt = P.external_influences(content)
# print(INSTRUCTION, "\n", external_influences_prompt)
response = R.external_influences()
# print(response)
parsed_response = Parsers.parse_json(response)
# print(json.dumps(parsed_response, indent=4))
patch = V.external_influences(parsed_response)
QM.update_dict(content, patch)
print(len(content["details"]["external_influences"]),
      "external influences generated")

# recurse here on bootstrap for each external influence

#### ACTIVITIES ####
activity_groups_prompt = P.activity_groups(content)
# pyperclip.copy(INSTRUCTION + "\n" + activity_groups_prompt)
response = R.activity_groups()
parsed_response = Parsers.parse_table(response)
# print(json.dumps(parsed_response, indent=4))
patch = V.activity_groups(content, parsed_response)
QM.update_dict(content, patch)
# print(json.dumps(content["groups"]["activity"], indent=4))
print(len(content["groups"]["activity"]), "activity groups generated", "\n", sum(
    [int(g["population"]) for _, g in content["groups"]["activity"].items()]), "people in activity groups")


#### GENERATE POPULATION ####
patch = CP.population(content)
# # print(patch)
QM.update_dict(content, patch)
# # print(json.dumps(content["npcs"], indent=4))
print(len(content["groups"]["social"]), "social groups generated")
print(len(content["npcs"]), "npcs generated")
# # print(json.dumps(content["details"], indent=4))

# for g in content["groups"]["social"]:
#     print(content["groups"]["social"][g]["type"],
#           content["groups"]["social"][g]["population"])
# for g in content["groups"]["activity"]:
#     print(content["groups"]["activity"][g]["type"],
#           content["groups"]["activity"][g]["population"])
#### QUERY population details ####


#### QUERY prominent organization characters ####
#### DISTRIBUTE PROMINENT CHARACTERS AMONG NPCs ####


#### DISTRIBUTE NPCS AMONG ACTIVITY GROUPS ####
# patch = CP.activity_members(content)


# # dump content into file
# with open("content.json", "w") as f:
#     json.dump(content, f, indent=4)


In [ ]:
import numpy.random as npr
import random
import matplotlib.pyplot as plt

# %matplotlib inline
# plt.style.use('ggplot')

s = []
for i in range(100):
    ref = npr.normal(10, 4, 1)[0]
    if ref < 10:
        ref = (ref) / 1.8
    else:
        ref -= 4
    ref = max(1, ref)
    print(int(ref))
    s.append(ref)

s = sorted(s)

count, bins, ignored = plt.hist(s, 30, normed=True)

plt.show()


In [ ]:
import json

import query
import clean_prompt as P
from logger import Logger
from llm import ChatGPT
import defines as D
import data_formatter as DF

Logger = Logger()
FORCE_HUMAN_READABLE_CACHE = True
with open("openai.key", "r") as f:
    api_key = f.readline().strip()
    api_id = f.readline().strip()

LLM = ChatGPT(Logger, INSTRUCTION, model_name="gpt-3.5-turbo", api_key=api_key, api_id=api_id)
Q = query.LLM_Query(LLM, Logger)
activity_groups_prompt = P.activity_groups(content)
parsed_content, response = LLM.query(activity_groups_prompt, options={"id": "all", "type": "correction"})
print(parsed_content)


In [ ]:
# Import the libraries
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

# Define the list of people and their bonds
people = ["Alice", "Bob", "Charlie", "David", "Eve", "Frank", "Grace", "Harry"]
bonds = [["blood", "interests", "business"], ["blood", "place"], ["interests", "business"], ["enemy", "cause"], ["business", "place"], ["interests", "cause"], ["blood", "enemy"], ["place", "cause"]]

# Define the weights for each bond
weights = {"blood": 0.9, "interests": 0.8, "business": 0.7, "enemy": -0.9, "cause": 0.6, "identity": 0.5, "ideology": 0.4, "past": 0.3, "knowledge": 0.2, "place": 0.1, "friendship": 0.05}

# Define the existing groups and their keywords
groups = {"A": ["blood", "place"], "B": ["interests", "business"], "C": ["enemy", "cause"]}

# Convert the bonds to a matrix of features
features = np.zeros((len(people), len(weights)))
for i, bond in enumerate(bonds):
    for b in bond:
        features[i][list(weights.keys()).index(b)] = weights[b]

# Convert the groups to a matrix of features
group_features = np.zeros((len(groups), len(weights)))
for i, group in enumerate(groups.values()):
    for g in group:
        group_features[i][list(weights.keys()).index(g)] = weights[g]

# Compute the similarity matrix between people and groups
similarity = cosine_similarity(features, group_features)

# Apply K-Means with precomputed distances
clustering = KMeans(n_clusters=len(groups)).fit(similarity)

# Print the results
print("People:", people)
print("Bonds:", bonds)
print("Groups:", list(groups.keys()))
print("Assignments:", clustering.labels_)


In [ ]:
# Import the libraries
import numpy as np
from k_means_constrained import KMeansConstrained
from sklearn.metrics.pairwise import cosine_similarity

# Define the list of people and their bonds
people = ["Alice", "Bob", "Charlie", "David", "Eve", "Frank", "Grace", "Harry"]
bonds = [["blood", "interests", "business"], ["blood", "place"], ["interests", "business"], ["enemy", "cause"], ["business", "place"], ["interests", "cause"], ["blood", "enemy"], ["place", "cause"]]

# Define the weights for each bond
weights = {"blood": 0.9, "interests": 0.8, "business": 0.7, "enemy": 0.9, "cause": 0.6, "identity": 0.8, "ideology": 0.7, "past": 0.8, "knowledge": 0.8, "place": 0.7, "friendship": 0.8}

# Define the existing groups and their keywords
groups = {"A": ["blood", "place"], "B": ["interests", "business"], "C": ["enemy", "cause"]}

# Define the fixed size for each group
group_size = 3

# Convert the bonds to a matrix of features
features = np.zeros((len(people), len(weights)))
for i, bond in enumerate(bonds):
    for b in bond:
        features[i][list(weights.keys()).index(b)] = weights[b]

# Convert the groups to a matrix of features
group_features = np.zeros((len(groups), len(weights)))
for i, group in enumerate(groups.values()):
    for g in group:
        group_features[i][list(weights.keys()).index(g)] = weights[g]

# Compute the similarity matrix between people and groups
similarity = cosine_similarity(features, group_features)

# Apply K-Means Constrained with precomputed distances and size constraints
clustering = KMeansConstrained(n_clusters=len(groups), size_min=1, size_max=group_size).fit(similarity)

# Print the results
print("People:", people)
print("Bonds:", bonds)
print("Groups:", list(groups.keys()), list(groups.values()))
print("Assignments:", clustering.labels_)


In [ ]:
# Import the libraries
import numpy as np
from k_means_constrained import KMeansConstrained
from nltk.tokenize import word_tokenize
import nltk

nltk.download("punkt")

# Define the list of people and their bonds
people = ["Alice", "Bob", "Charlie", "David", "Eve", "Frank", "Grace", "Harry"]
bonds = [["blood", "interests", "business"], ["blood", "place"], ["interests", "business"], ["enemy", "cause"], ["business", "place"], ["interests", "cause"], ["blood", "enemy"], ["place", "cause"]]

# Define the existing groups and their keywords
groups = {"A": ["blood", "place"], "B": ["interests", "business"], "C": ["enemy", "cause"], "D": ["place", "cause"], "E": ["enemy", "interests"]}

# Define the fixed size for each group
group_size = 3

# Convert the bonds to a list of tokens
tokens = []
for bond in bonds:
    tokens.append(word_tokenize(" ".join(bond)))

# Convert the groups to a list of tokens
group_tokens = []
for group in groups.values():
    group_tokens.append(word_tokenize(" ".join(group)))

# # Compute the token overlap matrix between people and groups
# overlap = np.zeros((len(people), len(groups)))
# for i, token in enumerate(tokens):
#     for j, group_token in enumerate(group_tokens):
#         overlap[i][j] = len(set(token).intersection(set(group_token)))

# Compute the Jaccard similarity matrix between people and groups
jaccard = np.zeros((len(people), len(groups)))
for i, token in enumerate(tokens):
    for j, group_token in enumerate(group_tokens):
        jaccard[i][j] = len(set(token).intersection(set(group_token))) / len(set(token).union(set(group_token)))

# Apply K-Means Constrained with precomputed distances and size constraints
# clustering = KMeansConstrained(n_clusters=len(groups), size_min=1, size_max=group_size).fit(overlap)
clustering = KMeansConstrained(n_clusters=len(groups), size_min=1, size_max=group_size).fit(jaccard)

# Print the results
print("People:", people)
print("Bonds:", bonds)
print("Groups:", list(groups.keys()), list(groups.values()))
print("Assignments:", clustering.labels_)


In [ ]:
# Define the list of people and their bonds
people = ["Alice", "Bob", "Charlie", "David", "Eve", "Frank", "Grace", "Harry"]
bonds = [["blood", "interests", "business"], ["blood", "place"], ["interests", "business"], ["enemy", "cause"], ["business", "place"], ["interests", "cause"], ["blood", "enemy"], ["place", "cause"]]

# Define the existing groups and their keywords
groups = {"A": ["blood", "place"], "B": ["interests", "business"], "C": ["enemy", "cause"], "D": ["place", "cause"], "E": ["enemy", "interests"]}

# Convert the bonds to a list of tokens
tokens = []
for bond in bonds:
    tokens.append(word_tokenize(" ".join(bond)))

# Convert the groups to a list of tokens
group_tokens = []
for group in groups.values():
    group_tokens.append(word_tokenize(" ".join(group)))

jaccard = np.zeros((len(people), len(groups)))
for i, token in enumerate(tokens):
    for j, group_token in enumerate(group_tokens):
        jaccard[i][j] = len(set(token).intersection(set(group_token))) / len(set(token).union(set(group_token)))

# print(jaccard)
# for i, v in enumerate(jaccard):
#     group_chosen = list(groups.keys())[np.argmax(v)]
#     print(people[i], group_chosen, groups[group_chosen], bonds[i])


# Compute the token overlap matrix between people and groups
overlap = np.zeros((len(people), len(groups)))
for i, token in enumerate(tokens):
    for j, group_token in enumerate(group_tokens):
        overlap[i][j] = len(set(token).intersection(set(group_token)))

print(overlap)


In [ ]:
# Import the libraries
import nltk
from nltk.corpus import wordnet

# Download WordNet data
nltk.download("wordnet")

# Define two words to compare
word1 = "temple"
word2 = "cult"

# Get the synsets of the words
synsets1 = wordnet.synsets(word1)
synsets2 = wordnet.synsets(word2)

# Choose the first sense of each word
sense1 = synsets1[0]
sense2 = synsets2[0]

# Compute the path similarity between the senses
similarity = sense1.path_similarity(sense2)

# Print the result
print(f"The similarity between {word1} and {word2} is {similarity}")


In [ ]:
import torch
import torchtext

# Load pre-trained GloVe embeddings
glove = torchtext.vocab.GloVe(name="6B", dim=50)

people = ["Alice", "Bob", "Charlie", "David", "Eve", "Frank", "Grace", "Harry"]
bonds = [["blood", "interests", "business"], ["blood", "tree"], ["interests", "business"], ["enemy", "cause"], ["business", "place"], ["interests", "cause"], ["blood", "enemy"], ["place", "cause"]]

# Define the existing groups and their keywords
groups = {"A": ["blood", "place"], "B": ["interests", "business"], "C": ["enemy", "cause"], "D": ["hemoglobin", "plant"], "E": ["enemy", "interests"]}

for i in range(len(people)):
    group1 = bonds[i]
    for k, v in groups.items():
        group2 = v
        # Convert each word to its embedding vector
        group1_vectors = [glove[word] for word in group1]
        group2_vectors = [glove[word] for word in group2]
        # Use unsqueeze() to add a singleton dimension to group1_vectors
        group1_vectors = torch.stack(group1_vectors).unsqueeze(1)  # group1_vectors is now of size (3, 1, 50)
        # Use cosine_similarity() to compute the similarity along the last dimension
        similarity_matrix = torch.cosine_similarity(group1_vectors, torch.stack(group2_vectors), dim=-1)  # similarity_matrix is of size (3, 2)
        average_similarity = torch.mean(similarity_matrix)
        # Alternatively, calculate the maximum cosine similarity between each pair of vectors
        max_similarity = torch.max(similarity_matrix)

        print(people[i], k, average_similarity, max_similarity, group1, group2)


In [ ]:
import nltk
from nltk.corpus import wordnet as wn

people = ["Alice", "Bob", "Charlie", "David", "Eve", "Frank", "Grace", "Harry"]
bonds = [["blood", "interests", "business"], ["blood", "tree"], ["interests", "business"], ["enemy", "cause"], ["business", "place"], ["interests", "cause"], ["blood", "enemy"], ["place", "cause"]]

# Define the existing groups and their keywords
groups = {"A": ["blood", "garbage"], "B": ["interests", "money"], "C": ["enemy", "cause"], "D": ["hemoglobin", "plant"], "E": ["enemy", "interests"]}

# # Define two groups of words
# group1 = ["cat", "dog", "pet", "animal"]
# group2 = ["car", "bike", "vehicle", "transport"]

for i in range(len(people)):
    group1 = bonds[i]
    for k, v in groups.items():
        group2 = v

        # Convert each word to its first synset
        group1_synsets = [wn.synsets(word)[0] for word in group1]
        group2_synsets = [wn.synsets(word)[0] for word in group2]

        # Calculate the average path similarity between each pair of synsets
        similarity_matrix = [[synset1.path_similarity(synset2) for synset2 in group2_synsets] for synset1 in group1_synsets]
        average_similarity = sum(sum(row) for row in similarity_matrix) / (len(group1) * len(group2))
        # print(average_similarity) # 0.16666666666666666

        # Alternatively, calculate the maximum path similarity between each pair of synsets
        max_similarity = max(max(row) for row in similarity_matrix)
        # print(max_similarity) # 0.3333333333333333

        print(people[i], k, average_similarity, max_similarity, group1, group2)


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

# Load pre-trained BERT tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
model = AutoModel.from_pretrained("bert-base-multilingual-cased")

people = ["Alice", "Bob", "Charlie", "David", "Eve", "Frank", "Grace", "Harry"]
bonds = [["blood", "interests", "business"], ["blood", "tree"], ["interests", "business"], ["enemy", "cause"], ["business", "place"], ["interests", "cause"], ["blood", "enemy"], ["place", "cause"]]

# Define the existing groups and their keywords
groups = {"A": ["blood", "garbage"], "B": ["interests", "money"], "C": ["cause", "enemy"], "D": ["hemoglobin", "plant"], "E": ["enemy", "interests"]}


for i in range(len(people)):
    chosen_groups = []
    group1 = bonds[i]
    for k, v in groups.items():
        group2 = v
        # Convert each word to its embedding vector using BERT
        group1_vectors = []
        group2_vectors = []
        for word in group1 + group2:
            # Tokenize the word and add special tokens [CLS] and [SEP]
            input_ids = torch.tensor([tokenizer.encode(word, add_special_tokens=True)])
            # Get the last hidden state of the token [-1] from the BERT model
            with torch.no_grad():
                output = model(input_ids)
            last_hidden_state = output[0]
            # Take the mean of the last hidden state to get a single vector for the word
            mean_last_hidden_state = torch.mean(last_hidden_state, dim=1)
            # Append the vector to the corresponding list
            if word in group1:
                group1_vectors.append(mean_last_hidden_state)
            else:
                group2_vectors.append(mean_last_hidden_state)

        if set(sorted(group1)) == set(sorted(group2)):
            # Use a default value of 1.0 as the similarity
            average_similarity = 1.0
            max_similarity = 1.0
        else:
            # Use unsqueeze() to add a singleton dimension to group1_vectors
            group1_vectors = torch.stack(group1_vectors).unsqueeze(1)  # group1_vectors is now of size (3, 1, 768)
            # Use cosine_similarity() to compute the similarity along the last dimension
            similarity_matrix = torch.cosine_similarity(group1_vectors, torch.stack(group2_vectors), dim=-1)  # similarity_matrix is of size (3, 2)
            average_similarity = torch.mean(similarity_matrix)
            # Alternatively, calculate the maximum cosine similarity between each pair of vectors
            max_similarity = torch.max(similarity_matrix)

        chosen_groups.append((k, average_similarity, max_similarity))
        # print(people[i], k, average_similarity, max_similarity, group1, group2)
    chosen_groups = sorted(chosen_groups, key=lambda x: x[1], reverse=True)
    chosen_groups = sorted(chosen_groups, key=lambda x: x[2], reverse=True)
    # print(people[i], chosen_groups[0][0], chosen_groups[0][1], chosen_groups[0][2], group1, groups[chosen_groups[0][0]])
    print(people[i], chosen_groups)


# clustering = KMeansConstrained(n_clusters=len(groups), size_min=1, size_max=group_size).fit(jaccard)

# # Print the results
# print("People:", people)
# print("Bonds:", bonds)
# print("Groups:", list(groups.keys()), list(groups.values()))
# print("Assignments:", clustering.labels_)


In [ ]:
from transformers import BertModel, BertTokenizer

# load pretrained model and tokenizer
model = BertModel.from_pretrained("bert-base-uncased")
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# encode lists as tensors
lists = bonds.copy()
encoded_lists = []
for lst in lists:
    # join keywords into a single string
    text = " ".join(lst)
    # tokenize and encode text
    input_ids = tokenizer.encode(text, add_special_tokens=True)
    # convert to tensor
    input_ids = torch.tensor([input_ids])
    # get BERT vector as the last hidden state of the [CLS] token
    with torch.no_grad():
        output = model(input_ids)
        bert_vector = output[0][:, 0, :]
    # append to encoded lists
    encoded_lists.append(bert_vector)

import torch
from torch import cosine_similarity

# calculate cosine similarity matrix
similarity_matrix = []
for v1 in encoded_lists:
    row = []
    for v2 in encoded_lists:
        # calculate cosine similarity between two vectors
        sim = cosine_similarity(v1, v2).item()
        # append to row
        row.append(sim)
    # append to similarity matrix
    similarity_matrix.append(row)

# convert to numpy array
similarity_matrix = np.array(similarity_matrix)


# apply spectral clustering
# from sklearn.cluster import SpectralClustering
# sc = SpectralClustering(n_clusters=3, affinity='precomputed')
# labels = sc.fit_predict(similarity_matrix)


from k_means_constrained import KMeansConstrained

clustering = KMeansConstrained(n_clusters=3, size_min=1, size_max=3, n_init=100, max_iter=1000, tol=0.0001).fit(similarity_matrix)

# # Print the results
# print("People:", people)
# print("Bonds:", bonds)
# print("Groups:", list(groups.keys()), list(groups.values()))
# print("Assignments:", clustering.labels_)

# print labels for each list
# for i, lst in enumerate(lists):
#     print(people[i], lst, clustering.labels_[i])

# order cluster by hash based on their people
clusters = {}
for i, lst in enumerate(lists):
    cluster = clustering.labels_[i]
    if cluster not in clusters:
        clusters[cluster] = []
    clusters[cluster].append(people[i])
for cluster in clusters:
    clusters[cluster] = sorted(clusters[cluster])
clusters = sorted(clusters.items(), key=lambda x: hash(tuple(x[1])))
clusters = [x[0] for x in clusters]
# print(clusters)

# print people per cluster
index = 1
for i in clusters:
    print("Cluster", index)
    for j, lst in enumerate(lists):
        if clustering.labels_[j] == i:
            print(people[j], lst)
    index += 1
    print()


In [ ]:
labels = ["A", "B", "C", "D", "E"]
size_limits = [2, 1, 6, 2, 4]
objects = ["a", "b", "c", "d", "e", "f", "g", "h", "i", "j"]
scores = [
    [1, 0.5, 0.8, 6, 0.1],
    [0.3, 0.2, 0.9, 0.4, 0.6],
    [0.5, 0.7, 0.2, 0.3, 0.4],
    [0.1, 0.3, 0.5, 0.7, 0.9],
    [0.2, 0.4, 0.6, 0.8, 1],
    [1, 1, 1, 1, 1],
    [2, 2, 2, 2, 2],
    [3, 3, 3, 3, 3],
    [4, 4, 4, 4, 4],
    [5, 5, 5, 5, 5],
]

label_counts = {label: 0 for label in labels}
object_labels = {}

for i in range(len(objects)):
    object_scores = scores[i]
    max_score = max(object_scores)
    max_index = object_scores.index(max_score)
    label = labels[max_index]
    while label_counts[label] >= size_limits[max_index]:
        object_scores[max_index] = -1
        max_score = max(object_scores)
        max_index = object_scores.index(max_score)
        label = labels[max_index]
    object_labels[objects[i]] = label
    label_counts[label] += 1

for object in objects:
    print(object, object_labels[object], scores[objects.index(object)])
    # print(object, object_labels[object])
# print(object_labels)


In [ ]:
from pulp import *

import random
import string

num_labels = 50
num_objects = 150

labels = [str(i) for i in range(1, num_labels + 1)]
size_limits = [random.randint(1, 5) for _ in range(num_labels)]
while sum(size_limits) < num_objects:
    size_limits = [random.randint(1, 5) for _ in range(num_labels)]

objects = ["_" + str(i) for i in range(num_objects)]
scores = [[random.random() for _ in range(num_labels)] for _ in range(num_objects)]

prob = LpProblem("Label Assignment", LpMaximize)

x = LpVariable.dicts("x", (range(len(objects)), range(len(labels))), cat="Binary")

prob += lpSum([scores[i][j] * x[i][j] for i in range(len(objects)) for j in range(len(labels))])

for i in range(len(objects)):
    prob += lpSum([x[i][j] for j in range(len(labels))]) == 1

for j in range(len(labels)):
    prob += lpSum([x[i][j] for i in range(len(objects))]) <= size_limits[j]

prob.solve()

object_labels = {}
for i in range(len(objects)):
    for j in range(len(labels)):
        if x[i][j].varValue == 1:
            object_labels[objects[i]] = labels[j]

# print labels and their objects
for i, label in enumerate(labels):
    selected = []
    for j, object in enumerate(objects):
        if object_labels[object] == label:
            score = scores[j][i]
            score_list = scores[j].copy()
            sorted_score_list = sorted(score_list, reverse=True)
            rank = sorted_score_list.index(score) + 1
            selected.append((object, rank))
    print(label, size_limits[i], selected)

    # print(label, size_limits[i], [object for object in objects if object_labels[object] == label])
#     # for each object, print the rank in the list of scores for this label score

#     score_list = scores[obj_index]
#     sorted_score_list = sorted(score_list, reverse=True)
#     rank = sorted_score_list.index(score_list[labels.index(label)]) + 1

# label_objects = {label: [] for label in labels}
# for obj, label in object_labels.items():
#     label_objects[label].append(obj)

# for label, objects in label_objects.items():
#     print(f"Label {label}:")
#     print("Objects:", objects)
#     for obj in objects:
#         obj_index = ord(obj) - ord('a')
#         score_list = scores[obj_index]
#         sorted_score_list = sorted(score_list, reverse=True)
#         rank = sorted_score_list.index(score_list[labels.index(label)]) + 1
#         print(f"Object {obj} rank: {rank}")


# print(object_labels)

# for object in objects:
# print(object, object_labels[object])#, scores[objects.index(object)])
